# 5 Colab / GPU: the real experiments

The laptop-scale configs validate the code but starve each expert of gradient steps, so their accuracy numbers are noise. This notebook runs the full-scale matrix on a GPU.

**Set the runtime to GPU first:** Runtime → Change runtime type → T4 (or better).

| | per run | tier1 (~15 runs) |
|---|---|---|
| T4 | ~1–1.5 h | ~15–20 h |
| A100 | ~30 min | ~6–8 h |
| laptop (MPS) | ~7 h | not viable |


## Setup


In [ ]:
!nvidia-smi -L || echo 'NO GPU - switch the runtime before continuing'


In [ ]:
REPO = 'https://github.com/depetrofabio/EnsembleFedLearning.git'

import os, pathlib
if not pathlib.Path('EnsembleFedLearning').exists():
    !git clone -q $REPO
%cd EnsembleFedLearning
!pip install -q -r requirements.txt
!python -m hefl.test_invariants


## Optional: persist results to Drive

Colab wipes the filesystem when the session ends. Mount Drive if you want the runs to survive.


In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    out_root = '/content/drive/MyDrive/hefl_results'
    !mkdir -p $out_root && rm -rf hefl/results && ln -s $out_root hefl/results
    print('results -> Drive')
else:
    print('results stay in the Colab session (lost on disconnect)')


## One full run

Start here to confirm the timing on your GPU before launching the whole matrix.


In [ ]:
%%time
!python -u -m hefl.run --config hefl/configs/rotation_dirichlet.json \
    --seed 42 --centralized --output_dir ./hefl/results/main_seed42


In [ ]:
print(open('hefl/results/main_seed42/table.md').read())


## The full matrix

`tier1` is the minimum a reviewer accepts: the main setting on 3 seeds, both single-axis controls, and the `random` / `oracle` assignment ablations.

The **random-assignment control is not optional**  *K* models beating one model is a capacity confound until random assignment is shown to be worse.

The script skips any run that already has a `table.md`, so it survives a disconnect: just re-run the cell.


In [ ]:
!bash scripts/run_sweep.sh tier1


In [ ]:
# ablations: number of clusters, α, clustering signal, dense mode, logit adjustment
!bash scripts/run_sweep.sh tier2


## Aggregate


In [ ]:
!python -m hefl.aggregate
print(open('hefl/results/_summary/main_table.md').read())


## Download the results

Only the small artefacts — checkpoints are excluded, so this stays a few hundred KB.


In [ ]:
!find hefl/results -name '*.json' -o -name '*.md' | grep -v models | tar -czf hefl_results.tar.gz -T -
from google.colab import files; files.download('hefl_results.tar.gz')
